In [ ]:
import random
import numpy as np
import time
import numba
import math
from numpy.random import Generator, PCG64DXSM, SeedSequence
import multiprocessing as mp
import concurrent.futures
from structure_FNs_base_2026 import play_sequence

np.set_printoptions(suppress=True)

In [ ]:
# use this cell for setting initial perameters

sides = 1
terms = 7
maxvalue = 10
startstop = 2
noise = 0.
annealing = 0.

timesteps = 10**8
runs = 1000

rein1 = 4
rein2 = 4
punish1 = -11
punish2 = -11
inertia = 1

nsteps = 100
blocklength = nsteps * 10
nsteps_eval = nsteps * 50 * 7 # Example value for all-pairs evaluation

def make_linear_pairs(n_stim, hold_out=None):
    pairs = []
    for i in range(n_stim - 1):
        pairs.append([i, i+1])
        pairs.append([i+1, i])
    
    nonadjpairs = []
    for i in range(n_stim):
        for j in range(i + 2, n_stim):
            nonadjpairs.append([i, j])
            nonadjpairs.append([j, i])
            
    # Default test pair: B-D and D-B analogue
    testpairs = np.asarray([[1, 3], [3, 1]])
    pairs = np.asarray(pairs)
    nonadjpairs = np.asarray(nonadjpairs)
    testpairs = np.asarray(testpairs)
    
    # Reward arrays
    pairs_reward = np.asarray([[0,1] if a < b else [1,0] for a,b in pairs], dtype=np.int64)
    nonadjpairs_reward = np.asarray([[0,1] if a < b else [1,0] for a,b in nonadjpairs], dtype=np.int64)
    testpairs_reward = np.asarray([[0,1] if a < b else [1,0] for a,b in testpairs], dtype=np.int64)
    
    return pairs, nonadjpairs, testpairs, pairs_reward, nonadjpairs_reward, testpairs_reward

"""
def make_circular_pairs(n_stim):
    # For a pair x, y with y>x, let d_x = y -x, let d_y = (x-y)%terms, 
    # then reward y if d_x < d_y and reward x if d_x > d_y.
    pairs = []
    for i in range(n_stim):
        j = (i + 1) % n_stim
        pairs.append([i, j])
        pairs.append([j, i])
    
    nonadjpairs = []
    for i in range(n_stim):
        for j in range(i + 2, n_stim):
            if i == 0 and j == n_stim - 1: continue # Already adjacent
            nonadjpairs.append([i, j])
            nonadjpairs.append([j, i])
            
    testpairs = np.asarray([[1, 3], [3, 1]]) # Dummy
    pairs = np.asarray(pairs)
    nonadjpairs = np.asarray(nonadjpairs)
    testpairs = np.asarray(testpairs)
    
    # Circular reward logic
    def circular_reward(a, b, n):
        dx = (b - a) % n
        dy = (a - b) % n
        if dx < dy: return [0, 1] # reward b
        else: return [1, 0] # reward a
    
    pairs_reward = np.asarray([circular_reward(a, b, n_stim) for a,b in pairs], dtype=np.int64)
    nonadjpairs_reward = np.asarray([circular_reward(a, b, n_stim) for a,b in nonadjpairs], dtype=np.int64)
    testpairs_reward = np.asarray([circular_reward(a, b, n_stim) for a,b in testpairs], dtype=np.int64)
    
    return pairs, nonadjpairs, testpairs, pairs_reward, nonadjpairs_reward, testpairs_reward
"""
            

In [ ]:
# run the code
start = time.perf_counter()

pairs, nonadjpairs, testpairs, pairs_reward, nonadjpairs_reward, testpairs_reward = make_linear_pairs(terms)
allpairs = np.concatenate((pairs, nonadjpairs, testpairs))
allpairs_reward = np.concatenate((pairs_reward, nonadjpairs_reward, testpairs_reward))

plen = len(pairs)
alen = len(allpairs)

sg = SeedSequence()
rgs = [Generator(PCG64DXSM(s)) for s in sg.spawn(runs)]

args = []
for n in range(runs):
    args.append((n, rgs[n], rein1, punish1, rein2, punish2, timesteps, nsteps, sides, pairs, testpairs, nonadjpairs, allpairs, plen, alen, terms, maxvalue, startstop, noise, annealing, runs, inertia, blocklength, pairs_reward, nsteps_eval))

mpseq_final = []
with concurrent.futures.ProcessPoolExecutor(max_workers=mp.cpu_count()) as executor:
    future_to_run = {executor.submit(play_sequence, *arg): arg for arg in args}
    for i, future in enumerate(concurrent.futures.as_completed(future_to_run)):
        try:
            res = future.result()
            mpseq_final.append(res)
            if (i + 1) % 100 == 0:
                print(f'Finished run #{i+1}')
        except Exception as exc:
            print(f'Run {i} generated an exception: {exc}')

final_sigweights = np.asarray([res[0] for res in mpseq_final])
final_cumsuc = np.asarray([res[1] for res in mpseq_final])
final_cumsucadd = np.asarray([res[2] for res in mpseq_final])
final_testcumsucadd = np.asarray([res[3] for res in mpseq_final])
final_recweights = np.asarray([res[4] for res in mpseq_final])
final_test_pair_stats = np.asarray([res[5] for res in mpseq_final])
final_all_pair_stats = np.asarray([res[6] for res in mpseq_final] if res[6] is not None else [None]*runs)

print("average cumsuc = ")
print(np.average(final_cumsuc)/runs)
print(" ")
print("average final cumsucadd = ")
print(np.average(final_cumsucadd)/(100))
print(" ")print("average test cumsum = ")
print(np.average(final_testcumsucadd)/(100))
print(" ")
finish = time.perf_counter()
print(f'Finished in {round(finish-start,0)/60} minutes')

In [ ]:
np.save('new_structure_noiseAnn_dsktp_1s_BASElearning-MV10_sigweights', final_sigweights)
np.save('new_structure_noiseAnn_dsktp_1s_BASElearning-MV10_recweights', final_recweights)
np.save('new_structure_noiseAnn_dsktp_1s_BASElearning-MV10_testcumsucadd', final_testcumsucadd)
